# Token Usage

Tokens are how Gemini bills you, so it helps to know the two places you can see them.

**Before the call — `client.models.count_tokens()`**
Estimates how large your input is *without* generating anything. Use it to check a prompt
fits your budget before you pay for output.

**After the call — `response.usage_metadata`**
Breaks down what you were actually charged for: input tokens, output tokens, and — on
thinking models — the reasoning tokens spent before the answer was written.

In [ ]:
from google import genai
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"var: ")

_set_env("GEMINI_API_KEY")

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
prompt = "The quick brown fox jumps over the lazy dog."

# Count tokens before sending. `.total_tokens` is the count itself —
# printing the response object would dump the raw HTTP wrapper too.
count = client.models.count_tokens(
    model="gemini-3.8-flash", contents=prompt
)
print("total_tokens: ", count.total_tokens)
# ( e.g., total_tokens: 11 )

response = client.models.generate_content(
    model="gemini-3.8-flash", contents=prompt
)

# The usage_metadata provides detailed token counts.
print(response.usage_metadata)

## Watch `thoughts_token_count`

Notice the `thoughts_token_count` field in the metadata above. On a thinking model the
tokens spent reasoning are billed as output, and they are frequently **larger than the
answer itself** — one sample run of this notebook produced 153 thinking tokens against
only 62 tokens of visible answer.

So `prompt_token_count` plus the length of `response.text` will understate your bill.
Read `total_token_count` if you want the real number.